# OLIVE FRP: subject-level transfer for target vs. non-target decoding

A practical question for any new OLIVE user is: **before we ever calibrate a decoder on
this specific person, how well does a target-vs-non-target EEG decoder trained on *other*
participants generalize to them?** This is exactly the situation OLIVE's implicit
(gaze+EEG) channel faces the first time it sees a new user, before any subject-specific
calibration data exists.

This notebook trains a simple, dependency-light target-vs-non-target decoder using
**leave-one-subject-out (LOSO) cross-validation**: for each held-out participant, the
model is trained only on epochs from every *other* participant, then evaluated on the
held-out participant's epochs. The result is a per-subject **zero-shot transfer AUC**,
which is a much harder (and more realistic, for a brand-new user) test than a within-subject
train/test split.

We use the same `ApocalyVec/olive-frp` dataset as
[`erp_target_vs_nontarget.ipynb`](./erp_target_vs_nontarget.ipynb); see that notebook and
`release/dataset/CARD.md` for the full field documentation.

In [1]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend; this notebook saves a PNG instead of relying on a GUI
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

RNG_SEED = 0
np.random.seed(RNG_SEED)

## Load the dataset

Same local-parquet-first loading pattern as the first example notebook. We keep
`subject_id`, `user_study`, `task`, `y`, `eeg`, plus `p_target` (OLIVE's own online
default-decoder probability, used only for a reference comparison at the end).

In [2]:
_candidates = [
    Path("release/dataset/out/frp_dataset.parquet"),   # cwd == repo root
    Path("../dataset/out/frp_dataset.parquet"),          # cwd == release/examples/
    Path("dataset/out/frp_dataset.parquet"),             # cwd == release/
]
DATASET_PATH = next((p for p in _candidates if p.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        "Could not find frp_dataset.parquet next to this notebook. Run release/dataset/export_hf.py "
        "first, or point DATASET_PATH at your local copy."
    )
print(f"Loading {DATASET_PATH.resolve()}")

df = pd.read_parquet(DATASET_PATH)

# --- Alternative: load directly from the Hugging Face Hub ---
# from datasets import load_dataset
# hf_ds = load_dataset("ApocalyVec/olive-frp", split="train")
# df = hf_ds.to_pandas()

df = df[["subject_id", "user_study", "task", "y", "eeg", "p_target"]]
print(df.shape)
df.head()

Loading /Users/apocalyvec/PycharmProjects/rlpf/release/dataset/out/frp_dataset.parquet
(58184, 6)


## Epoch layout, and restricting to one task

Each `eeg` cell reshapes to a `[20, 230]` float32 array (20 channels, 256 Hz, `[-0.1, 0.8]` s
fixation-onset-locked window) exactly as in the first notebook. The corrected label
convention is `y == 1` -> target, `y == 0` -> non-target.

We restrict this notebook to **`task == "spaceshooter"`**. The companion ERP notebook shows
that SpaceShooter produces a much larger, more reliable target-vs-non-target amplitude
difference than Visual Search (roughly +12 uV vs. +0.4 uV at Pz, 300-500 ms), i.e. a
stronger signal for a decoder to pick up on, so it is the better task for a first
cross-subject transfer demo. (The same pipeline works unchanged on `visual_search`, or on
both tasks pooled, by changing `TASK` below.)

`subject_id` is consistent across user studies (the same participant can appear in US1/US2/US3
under one `subject_id`), so we group by `subject_id` alone, pooling a participant's epochs
across every study they took part in, for the leave-one-*subject*-out split.

In [3]:
FS = 256                # Hz
T0 = -0.1                # s, epoch start (fixation onset - 0.1 s)
N_T = 230                 # samples/channel
TASK = "spaceshooter"

def t_to_idx(t):
    """Time (s) -> sample index within a [-0.1, 0.8] s, 256 Hz epoch."""
    return int(round((t - T0) * FS))

def reshape_eeg(row_val):
    """Nested per-channel object array -> float32 [20, 230] ndarray."""
    return np.array(row_val.tolist(), dtype=np.float32)

task_df = df[df["task"] == TASK].reset_index(drop=True)
print(f"{TASK}: {len(task_df)} epochs, {task_df['subject_id'].nunique()} subjects")

spaceshooter: 30603 epochs, 25 subjects


## Subject eligibility and an epoch cap for speed

Two practical filters, both applied before feature extraction:

1. **Eligibility**: LOSO folds are unstable if a held-out subject has almost no epochs of
   one class (the fold's AUC becomes noisy/undefined). We keep only subjects with **>= 40
   target and >= 40 non-target** `spaceshooter` epochs.
2. **Epoch cap**: to keep this notebook fast and to avoid a handful of high-trial-count
   subjects dominating both the training folds and the scaler/model fit, we cap each
   subject's epochs at **400 per class** (random subsample, fixed seed), rather than using
   all ~30k epochs. Increase `CAP` (or set it to `None`) to use more data at the cost of
   slower LOSO fitting.

In [4]:
N_MIN = 40   # minimum epochs per class for a subject to be included
CAP = 400    # max epochs per (subject, class) kept, for speed

counts = task_df.groupby(["subject_id", "y"]).size().unstack(fill_value=0)
eligible_subjects = counts[(counts.get(0, 0) >= N_MIN) & (counts.get(1, 0) >= N_MIN)].index
task_df = task_df[task_df["subject_id"].isin(eligible_subjects)].reset_index(drop=True)
print(f"eligible subjects (>= {N_MIN} epochs/class): {len(eligible_subjects)}")
print(f"epochs after eligibility filter: {len(task_df)}")

parts = []
for (_subject_id, _y), g in task_df.groupby(["subject_id", "y"]):
    if len(g) > CAP:
        g = g.sample(CAP, random_state=RNG_SEED)
    parts.append(g)
task_df = pd.concat(parts).sort_index().reset_index(drop=True)
print(f"epochs after capping at {CAP}/subject/class: {len(task_df)}")

eligible subjects (>= 40 epochs/class): 24
epochs after eligibility filter: 30587
epochs after capping at 400/subject/class: 15659


## Features: per-channel, per-window baseline-corrected amplitude

To stay dependency-light (no torch, no deep decoder), each epoch is reduced to a compact
hand-crafted feature vector rather than fed to a decoder as a raw time series:

1. Baseline-correct each epoch (subtract its own `[-0.1, 0]` s pre-fixation mean, per channel,
   the same convention as the ERP notebook).
2. For three post-fixation windows, **0.1-0.3 s, 0.3-0.5 s, 0.5-0.8 s** (chosen to span the
   SpaceShooter target-vs-non-target difference visible in the ERP notebook, which is a
   sustained deflection rather than a single sharp peak), take each channel's **mean and
   standard deviation** of amplitude within the window.

That gives 20 channels x 3 windows x 2 statistics = **120 features** per epoch. Standardization
(z-scoring each feature) is fit on the training folds only, inside the LOSO loop below, to
avoid leaking held-out-subject statistics into the scaler.

In [5]:
BASELINE_SLICE = slice(t_to_idx(-0.1), t_to_idx(0.0))
WINDOWS = [(0.1, 0.3), (0.3, 0.5), (0.5, 0.8)]
WIN_SLICES = [slice(t_to_idx(a), t_to_idx(b)) for a, b in WINDOWS]

eeg_stack = np.stack([reshape_eeg(v) for v in task_df["eeg"].values], axis=0)  # [N, 20, 230]
baseline = eeg_stack[:, :, BASELINE_SLICE].mean(axis=2, keepdims=True)
eeg_bc = eeg_stack - baseline

feats = np.concatenate(
    [eeg_bc[:, :, s].mean(axis=2) for s in WIN_SLICES]
    + [eeg_bc[:, :, s].std(axis=2) for s in WIN_SLICES],
    axis=1,
)
y = task_df["y"].values
subjects = task_df["subject_id"].values
uniq_subjects = np.sort(np.unique(subjects))
print(f"feature matrix: {feats.shape}  ({len(uniq_subjects)} subjects for LOSO)")

feature matrix: (15659, 120)  (24 subjects for LOSO)


## Leave-one-subject-out (LOSO) transfer decoding

For each held-out subject: fit a `StandardScaler` and a `LogisticRegression(class_weight=
"balanced")` on every *other* subject's epochs, then score the held-out subject's epochs
with ROC-AUC. `class_weight="balanced"` matters because per-subject target/non-target ratios
vary a fair amount (see the eligibility table above) and pooling many subjects' epochs
together does not automatically balance classes.

Chance-level AUC is **0.5**. This is a genuinely hard test: it is zero-shot cross-subject
transfer with no calibration data from the held-out person at all.

In [6]:
per_subject_auc = {}
for held in uniq_subjects:
    train_mask = subjects != held
    test_mask = subjects == held

    scaler = StandardScaler().fit(feats[train_mask])
    X_train = scaler.transform(feats[train_mask])
    X_test = scaler.transform(feats[test_mask])

    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X_train, y[train_mask])

    p_test = clf.predict_proba(X_test)[:, 1]
    per_subject_auc[int(held)] = roc_auc_score(y[test_mask], p_test)

auc_values = np.array(list(per_subject_auc.values()))
print(f"LOSO transfer AUC across {len(auc_values)} held-out subjects:")
print(f"  mean = {auc_values.mean():.3f}   sd = {auc_values.std():.3f}")
print(f"  range = [{auc_values.min():.3f}, {auc_values.max():.3f}]  (chance = 0.500)")
for sid in sorted(per_subject_auc):
    print(f"  subject {sid:>3d}: AUC = {per_subject_auc[sid]:.3f}")

LOSO transfer AUC across 24 held-out subjects:
  mean = 0.544   sd = 0.034
  range = [0.448, 0.596]  (chance = 0.500)
  subject   4: AUC = 0.588
  subject  12: AUC = 0.547
  subject  18: AUC = 0.537
  subject  20: AUC = 0.520
  subject  28: AUC = 0.580
  subject  29: AUC = 0.541
  subject  31: AUC = 0.576
  subject  33: AUC = 0.579
  subject  34: AUC = 0.539
  subject  35: AUC = 0.515
  subject  36: AUC = 0.554
  subject  37: AUC = 0.448
  subject  39: AUC = 0.510
  subject  40: AUC = 0.530
  subject  46: AUC = 0.514
  subject  47: AUC = 0.580
  subject  48: AUC = 0.568
  subject  49: AUC = 0.541
  subject  51: AUC = 0.503
  subject  52: AUC = 0.547
  subject  53: AUC = 0.518
  subject  54: AUC = 0.585
  subject  55: AUC = 0.536
  subject  59: AUC = 0.596


## Per-subject transfer AUC, sorted

Bar plot of each held-out subject's transfer AUC, sorted low to high, against the chance
line (0.5) and the across-subject mean.

In [7]:
order = sorted(per_subject_auc, key=per_subject_auc.get)
sorted_aucs = [per_subject_auc[s] for s in order]
colors = ["crimson" if a < 0.5 else "steelblue" for a in sorted_aucs]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(range(len(order)), sorted_aucs, color=colors)
ax.axhline(0.5, color="k", lw=1, ls="--", label="chance (0.5)")
ax.axhline(auc_values.mean(), color="darkorange", lw=1.5, label=f"mean = {auc_values.mean():.3f}")
ax.set_xticks(range(len(order)))
ax.set_xticklabels([str(s) for s in order], rotation=90, fontsize=7)
ax.set_xlabel("held-out subject_id")
ax.set_ylabel("LOSO transfer AUC")
ax.set_title(f"Subject-level transfer: target vs. non-target ({TASK}), LOSO logistic regression")
ax.legend(fontsize=8)
fig.tight_layout()

out_png = Path("subject_transfer_decoding.png")
fig.savefig(out_png, dpi=150)
print(f"saved {out_png.resolve()}")

saved /Users/apocalyvec/PycharmProjects/rlpf/release/examples/subject_transfer_decoding.png


## Reference point: OLIVE's own online decoder (`p_target`)

The dataset's `p_target` column is the probability OLIVE's actual *deployed*, per-subject
**default decoder** assigned to a fixation being on the target, logged during the live
sessions (`p_target_quality` flags its confidence; it is only populated for the sessions
where the online decoder ran). Unlike the LOSO model above, `p_target` reflects a decoder
that had within-subject information available (and was not doing zero-shot cross-subject
transfer), so this is not an apples-to-apples comparison. It is a rough ceiling reference
for "what a properly calibrated, deployed decoder looks like for this signal," not a
same-condition baseline.

In [8]:
p_target_aucs = []
for sid, g in task_df.groupby("subject_id"):
    valid = g["p_target"].notna()
    if valid.sum() < 10 or g.loc[valid, "y"].nunique() < 2:
        continue
    p_target_aucs.append(roc_auc_score(g.loc[valid, "y"], g.loc[valid, "p_target"]))

if p_target_aucs:
    p_target_aucs = np.array(p_target_aucs)
    print(f"OLIVE online default-decoder (p_target) AUC, {len(p_target_aucs)} subjects with logged p_target:")
    print(f"  mean = {p_target_aucs.mean():.3f}   sd = {p_target_aucs.std():.3f}")
else:
    print("No subjects with sufficient logged p_target in this task subset.")

print(f"\nFor comparison, this notebook's LOSO cross-subject transfer AUC: mean = {auc_values.mean():.3f}")

OLIVE online default-decoder (p_target) AUC, 10 subjects with logged p_target:
  mean = 0.746   sd = 0.120

For comparison, this notebook's LOSO cross-subject transfer AUC: mean = 0.544


## Caveats and next steps

- **This is a simple baseline, not a ceiling.** The feature set (per-channel window mean/std)
  and model (logistic regression) are chosen to be dependency-light (no `torch`, no GPU) and
  fast to run in a notebook, not to maximize accuracy. To try a stronger decoder, replace the
  "Features" and "LOSO" cells above with, e.g., an **EEGNet** (or other CNN) trained per-fold
  directly on the baseline-corrected `[20, 230]` epochs; the LOSO loop structure (train
  mask / test mask by `subject_id`, ROC-AUC per held-out subject) does not need to change.
- **Zero-shot transfer is a lower bound, not the deployed number.** OLIVE's actual online
  decoder (`p_target`, referenced above) calibrates per subject; a real deployment would
  typically use a short per-subject calibration phase (or few-shot fine-tuning on top of a
  cross-subject-pretrained model like the one above) rather than pure zero-shot transfer.
- **AUC varies a lot by subject** (see the bar plot): some held-out subjects transfer close to
  chance while others transfer noticeably better, which is itself useful information about
  which participants' target-related EEG signatures are more idiosyncratic.